In [ ]:
%matplotlib inline

# Version 9 mai 2026
import os, fnmatch, sys
import io, contextlib
import glob
import numpy as np
import numpy.ma as ma

import pandas as pd
import geopandas as gp
from pathlib import Path
from jupyprint import jupyprint  
from IPython.display import display, FileLink, FileLinks, HTML
from yaml import safe_load
from copy import copy

# matplotlib
from matplotlib import colors
from pylab import *
from IPython.display import Image
from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

from itertools import accumulate
from operator import add 

from tqdm.auto import tqdm  # au lieu de: from tqdm import tqdm
from yaml import safe_dump
from scipy.interpolate import RegularGridInterpolator
from shapely import LineString, intersection

################
# Clawpack     #
################
# Source : http://www.clawpack.org/gallery/_static/apps/notebooks/visclaw/gridtools.html#Extract-1d-transects
from clawpack.visclaw import colormaps, frametools, geoplot, gridtools
from clawpack.visclaw import animation_tools
from clawpack.visclaw.plottools import pcolorcells
from clawpack.geoclaw import dtopotools, topotools, marching_front
from clawpack.geoclaw import util
from clawpack.pyclaw.solution import Solution
from clawpack.pyclaw import solution as solution
from clawpack.amrclaw import region_tools
from clawpack.geoclaw import fgout_tools

################
# Latex        #
################
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['text.latex.preamble'] = r'\usepackage{libertine}'
plt.rcParams['font.size'] = 12

################
# Répertoires  #
################
current_directory = Path().cwd()
proj_dir      = current_directory.parent
topo_dir      = proj_dir / "Topo"
images_dir    = proj_dir / "Figures"
animation_dir = images_dir / 'Animation'
video_dir     = images_dir / 'Vidéos'
output_dir    = current_directory / "_output"
BC_dir        = proj_dir / "CL"
export_dir    = proj_dir / 'Export'
avac_dir      = proj_dir / "AVAC"
répertoires   = {'proj_dir':str(proj_dir),'topo_dir':str(topo_dir),
                 'avac_dir':str(avac_dir),'images_dir':str(images_dir),
                 'output_dir':str(output_dir),'waves_dir':str(current_directory)}
if not animation_dir.exists():
     animation_dir.mkdir(parents=True, exist_ok=True)
     print(f"Création du répertoire {animation_dir}")
if not video_dir.exists():
     video_dir.mkdir(parents=True, exist_ok=True)
     print(f"Création du répertoire {video_dir}")
HOME       = current_directory
répertoire_travail  = HOME 
répertoire_résultat = répertoire_travail / '_output'
configuration_file  = proj_dir / "impulse_configuration.yaml"
configuration_avac  = avac_dir / "AVAC_configuration.yaml"
sys.path.insert(0, str(proj_dir))

##################
# Configurations #
##################
if configuration_file.exists():
    print(f"Chargement du fichier de configuration existant")
    with open(configuration_file) as file:
        config        = safe_load(file)
        topo          = config["topo_files"]
        lake          = config["lake"]
        gauges        = config['gauges']
        rheology      = config['rheology']
        computation   = config['computation']
        output        = config['output']
        topo_files    = config['topo_files']
        scenario      = config['scenario']
        period_return = scenario['period_return']
        directory     = config['directory']
else:
    print(f"Le fichier {configuration_file} est manquant. Il faut le générer avec Waves_preprocess")
print('Répertoire où sont les données : \n  %s' % répertoire_résultat)
if not os.path.isdir(répertoire_travail):
      print('*** Revoir le nom du répertoire (je ne le trouve pas)')

if configuration_avac.exists():
    print(f"Chargement du fichier de configuration AVAC existant")
    with open(configuration_avac) as file:
        avac_config        = safe_load(file)
else:
    print(f"Le fichier {configuration_avac} est manquant. Il faut le générer avec le cahier AVAC")
config_total = {'avac':avac_config,'waves':config,'répertoires':répertoires}
################
# Module waves #
################
from module_waves import create_boundary_conditions, reload_wave, check_configuration, check_output, boundary_values
from module_waves import Create_Animation
from module_waves import calculate_overflow_rate, create_plot_lake_contour_meshing, calculate_inflow_rate

################
# AVAC         #
################
sys.path.insert(0, str(proj_dir / "AVAC"))
from module_avac import reading_raster_file_features, reload_avac, reading_raster_file, \
    export_claw_dem, export_claw_dem_window, determine_file_type, plot_topo, make_output
from module_avac import fn_eta, fn_ground, fn_h, fn_hu, fn_husquare, fn_hv, fn_u, fn_v, fn_extract
from module_avac import format_m
      
################
# Variables    #
################
CLAW           = os.environ['CLAW']
format_fichier = output['output_format']
rho            = rheology['rho']
g              = rheology['gravity']
z_lac          = lake['water_level'] if configuration_file.exists() else 1500
nb_grille      = computation['nb_grid'] # nombre d'éléments de la grille d'interpolation

topo_fine = reading_raster_file(str(topo_dir / topo_files['fine']))
xmin_mnt, xmax_mnt, ymin_mnt, ymax_mnt, nbx_mnt, nby_mnt, cell_size_mnt, dictionary_mnt, _, _,_ = \
        reading_raster_file_features(topo_dir / lake['topography'])
if check_configuration(config,topo_dir):
     print("Pas d'erreur détectée dans le fichier de configuration.")

avac_period = avac_config['release']['period_return']
wave_period = config['scenario']['period_return']
print(f"Période de retour considérée :\n * T= {avac_period} ans pour AVAC \n * T= {wave_period} ans pour Wave")
print(f"Répertoire de résultats : \n* AVAC : {avac_config['output']['output_directory']}\n* Wave : {output['output_directory']}")

# paramètre d'export des figures
figure_export_params = dict(dpi = 300, bbox_inches = 'tight')

# Topo

In [ ]:
# Ensemble des données topo
topo_file = reading_raster_file(topo_dir / lake['topography'])

fig, ax, x0, y0 = plot_topo(topo_file,step=500)
ax.add_patch(plt.Rectangle((lake['xmin']-x0,lake['ymin']-y0) , width=lake['xmax']-lake['xmin'], 
                           height=lake['ymax']-lake['ymin'], ls="-", lw=2, ec="red", fc="none"))

In [ ]:
# Ensemble des données topo et zoom sur le lac
crop_map       = True   # réduire la carte aux dimensions de la zone zoomée
step_plot_topo = config['output']['carto_layout']['minor_label_step']
text_m         = 10     # décalage du texte pour les étiquettes
margin         = config['output']['carto_layout']['margin']
crop_region    = (lake['xmin'] - margin, lake['xmax'] + margin,
                                 lake['ymin'] - margin, lake['ymax'] + margin)
if crop_map:
    topo_file_crop  = topo_file.crop(filter_region = crop_region)
else:
    topo_file_crop = copy(topo_file)
fig, ax, x0, y0 = plot_topo(topo_file_crop, step = step_plot_topo)
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=2, ec="red", fc="none"))

# Zoom sur le lac avec une marge

# ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
# ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)
if gauges['gauge_recording']:
    for k in range(len(gauges)-1):
        x = gauges[str(k)]['x']
        y = gauges[str(k)]['y']
        ax.scatter(x-x0,y-y0,c = 'red',marker='o',s=12)
        ax.text(x-x0,y-y0+text_m,str(1+k),color = 'red')

fig.savefig(images_dir / "vue_domaine_calcul.png",dpi = 300, bbox_inches = 'tight')

# Importation des résultats

In [ ]:
fichiers = fnmatch.filter(os.listdir(répertoire_résultat), 'fort.q*' )
NbSim    = len(fichiers)
if NbSim != computation['nb_simul']+1:
    print(f"Il y avait {computation['nb_simul']} simulations prévues, et je n'en trouve que {NbSim}.")
else:
    print(f"Il y a {NbSim} fichiers fort.q* (comme prévu).")


In [ ]:
frames = []
for i in range(NbSim):
      sol = solution.Solution(i, path=répertoire_résultat, file_format=format_fichier)
      frames.append(sol)

computation_domain, time_extent = check_output(frames)
dt  = time_extent['dt']
t_0 = time_extent['t_0']
t_f = time_extent['t_f']


In [ ]:
xmin, xmax = lake['xmin'], lake['xmax']
ymin, ymax = lake['ymin'], lake['ymax']

# grille
x = np.linspace(xmin,xmax,nb_grille)
y = np.linspace(ymin,ymax,nb_grille)
X_grille, Y_grille = np.meshgrid(x,y)
dx_grille = (xmax-xmin)/nb_grille
dy_grille = (ymax-ymin)/nb_grille

# interpolation de la solution 
sol_0     = gridtools.grid_output_2d(frames[0], fn_ground, X_grille, Y_grille, 
                                    levels='all',return_ma=True)
eta_0     = gridtools.grid_output_2d(frames[0], fn_eta, X_grille, Y_grille, 
                                      levels='all',return_ma=True)
hauteur_0 = gridtools.grid_output_2d(frames[0], fn_h, X_grille, Y_grille, 
                                      levels='all',return_ma=True)

volume_0 = int(hauteur_0.sum()*dx_grille*dy_grille)
print("La profondeur du lac est {:.2f} m.".format(hauteur_0.max()))
print(f"Le volume du lac est {format_m(volume_0)} m³." )

zmin = eta_0.min()
zmax = eta_0.max()
print(f"Cote minimale du MNT : {zmin:.1f} m. Cote maximale du MNT : {zmax:.1f} m.")
print(f"Cote maximale : {lake['water_level']} m.")


In [ ]:
# Tracé de la condition initiale
# contour rouge ->  domaine de calcul
# contour bleu -> masque de hauteur initiale mask_shp
lake_mask_gdf = gp.read_file(topo_dir / topo_files['mask_shp']) # importation du masque
step_plot_topo = config['output']['carto_layout']['minor_label_step']
if crop_map:
    fig, ax, x0, y0 = plot_topo(topo_file_crop, step=step_plot_topo)
else:
    fig, ax, x0, y0 = plot_topo(topo_file, step=step_plot_topo)
# limites du domaine
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=1, ec="red", fc="none"))

# Zoom sur le lac avec une marge
margin = 100  # mètres
ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)


#eta_masque = ma.masked_where(hauteur_0 < computation['dry_limit'], hauteur_0)
cmap_flat = ListedColormap([
                            (0,   0,   0,   0  ), # transparent
                            (0.2, 0.4, 1.0, 0.95) ]) # bleu])   
ax.imshow((hauteur_0 >= computation['dry_limit']).astype(float), origin='lower',
          extent=[xmin-x0, xmax-x0, ymin-y0, ymax-y0],
          cmap=cmap_flat, vmin=0, vmax=1)

# tracé du masque
lake_mask_gdf.geometry.translate(-x0, -y0).plot(
    ax=ax, facecolor='none', edgecolor='blue', linewidth=1.5 
)

 

#  Vérification des conditions aux limites

In [ ]:
temps_in    = np.arange(t_0,t_f+dt,dt)
versant     = 'est'
loc_légende = 'upper right'

fig, ((ax1, ax2,ax3)) = plt.subplots(1,3)
fig.suptitle(f"Flux moyen au cours du temps pour la frontière {versant}")
fig.set_figheight(4)
fig.set_figwidth(10)
## fig 1 ##
flux_masse_versant = -1*np.array([boundary_values(versant, frames[i], config)[3].mean() for i in range(0,NbSim)])
ax1.plot(temps_in, flux_masse_versant,label=r'$\dot M$')
ax1.set_ylabel(r"$\dot M$ [m$^2$/s]")

## fig 2 ##
hauteur_versant = np.array([boundary_values(versant, frames[i], config)[1].mean() for i in range(0,NbSim)])
ax2.plot(temps_in, hauteur_versant,label=r'$h$')
ax2.set_ylabel(r"$h$ [m]")

## fig 3 ##
vitesse_versant = -1*np.array([boundary_values(versant, frames[i], config)[2].mean() for i in range(0,NbSim)])
ax3.plot(temps_in, vitesse_versant,label=r'$v$')
ax3.set_ylabel(r"$v$ [m/s]")

for ax in (ax1,ax2,ax3):
    ax.grid()
    ax.legend(loc=loc_légende,ncol=2)
    ax.set_xlabel(r"$t$ [s]")
plt.tight_layout()

# Tracé de la condition initiale

In [ ]:
# Tracé de la condition initiale
# contour rouge ->  domaine de calcul
# contour bleu -> masque de hauteur initiale mask_shp
lake_mask_gdf = gp.read_file(topo_dir / topo_files['mask_shp']) # importation du masque
step_plot_topo = config['output']['carto_layout']['minor_label_step']
if crop_map:
    fig, ax, x0, y0 = plot_topo(topo_file_crop, step=step_plot_topo)
else:
    fig, ax, x0, y0 = plot_topo(topo_file, step=step_plot_topo)
# limites du domaine
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=1, ec="red", fc="none"))

# Zoom sur le lac avec une marge
margin = config['output']['carto_layout']['margin']  # mètres
ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)


#eta_masque = ma.masked_where(hauteur_0 < computation['dry_limit'], hauteur_0)
cmap_flat = ListedColormap([
                            (0,   0,   0,   0  ), # transparent
                            (0.2, 0.4, 1.0, 0.95) ]) # bleu])   
ax.imshow((hauteur_0 >= computation['dry_limit']).astype(float), origin='lower',
          extent=[xmin-x0, xmax-x0, ymin-y0, ymax-y0],
          cmap=cmap_flat, vmin=0, vmax=1)

# tracé du masque
lake_mask_gdf.geometry.translate(-x0, -y0).plot(
    ax=ax, facecolor='none', edgecolor='blue', linewidth=1.5 
)


## Énergie avalanche

In [ ]:
reload_wave()
from module_waves import create_plot_lake_contour_meshing, calculate_inflow_rate

In [ ]:
contour_dict = create_plot_lake_contour_meshing(config_total,topo_file = topo_file_crop, frame_test = 100)

In [ ]:
from module_waves import plot_avalanche
fig = plot_avalanche(45,config_total,topo_file_crop,verbosity=False)

In [ ]:
%matplotlib widget 
from module_waves import init_plot_avalanche_side_by_side, draw_avalanche_side_by_side

# ─────────────────────────────────────────────────────────
# Utilisation
ctx = init_plot_avalanche_side_by_side(config_total, topo_file_crop)

slider = widgets.SelectionSlider(
    options=[float(t) for t in ctx['wave_times']],
    description='t (s)',
    continuous_update=False,
    style={'description_width': 'initial'}
)
widgets.interactive(lambda t: draw_avalanche_side_by_side(t, ctx), t=slider)

In [ ]:
reload_wave()
from module_waves import init_plot_avalanche, draw_avalanche, calculate_inflow
 

In [ ]:
# Afficher 4*4 des infos
%matplotlib widget   
# à mettre en tête de notebook pour la mise à jour en place

import ipywidgets as widgets
from IPython.display import display

# création de la polyligne
pts_origine = [965800-1,6536300-1] 
polyligne = np.array([[0,0],[100,0],[100,-200]])

pts_origine = [965900-1,6536300-1] 
polyligne = np.array([[00,0],[00,-200]])
additionner = lambda P: [pts_origine[0]+P[0],pts_origine[1]+P[1]]
polyligne = LineString(list(map(additionner,polyligne)))


In [ ]:
# Comparaison des simulations en fonction du temps
ctx = init_plot_avalanche(config_total, topo_file_crop, polyligne)

slider = widgets.SelectionSlider(
    options=[float(t) for t in ctx['wave_times']],
    description='t (s)',
    continuous_update=False,
    style={'description_width': 'initial'}
)

out = widgets.interactive_output(
    lambda t: draw_avalanche(t, ctx), {'t': slider}
)

display(widgets.VBox([
    widgets.HBox([slider, ctx['info']]),
    out
]))

In [ ]:
# lecture du fichier de masque du lac
contour_dict = create_plot_lake_contour_meshing(config_total,topo_file = None)
# calcul des flux entrant dans le lac via son contour
frame_temps, flux_énergie_cinétique, flux_énergie_potentielle, wave_flux_volume, wave_volume_lac = calculate_inflow_rate(contour_dict,config_total)

In [ ]:
# intégration des puissances
énergie_cinétique_fournie_avalanche   = np.array(list(accumulate(flux_énergie_cinétique, add)))*dt
énergie_potentielle_fournie_avalanche = np.array(list(accumulate(flux_énergie_potentielle, add)))*dt
# calcul de l'énergie totale
flux_énergie                = np.array(flux_énergie_cinétique) + np.array(flux_énergie_potentielle)
énergie_fournie_avalanche   = np.array(énergie_cinétique_fournie_avalanche)+np.array(énergie_potentielle_fournie_avalanche)
# détermination de la durée de l'avalanche
scaled_flux = flux_énergie_cinétique/flux_énergie_cinétique.max()
ε = 1e-2 # seuil de détection
temps_arrivée_avalanche = frame_temps[scaled_flux> ε][0]
temps_fin_avalanche = frame_temps[scaled_flux> ε][-1]

fig, ((ax1, ax2)) = plt.subplots(1,2)
fig.suptitle(f"Puissance et énergie fournie par l'avalanche")
fig.set_figheight(4)
fig.set_figwidth(8)
ax1.plot(frame_temps,np.array(flux_énergie_cinétique)/1e6,label = r'$\dot E_c$')
ax1.plot(frame_temps,np.array(flux_énergie_potentielle)/1e6,label = r'$\dot E_p$')
ax1.plot(frame_temps,flux_énergie/1e6,'--k',label = r'$\dot E=\dot E_c+\dot E_p$')
ax1.set_ylabel(r"Puissance fournie au lac par l'avalanche [MW]")
ax1.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2)

ax2.plot(frame_temps,énergie_cinétique_fournie_avalanche/1e6,label = r'$ E_c$')
ax2.plot(frame_temps,énergie_potentielle_fournie_avalanche/1e6,label = r'$ E_p$')
ax2.set_ylabel(r"Énergie fournie par l'avalanche [MJ]")
ax2.plot(frame_temps,énergie_fournie_avalanche/1e6,'--k',label = r'$ E= E_c+ E_p$')
ax2.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2)

for ax in (ax1,ax2):
    ax.grid()
    ax.legend()
    ax.set_xlabel(r"$t$ [s]")
plt.tight_layout()
print(f"Flux avalancheux : t_début = {temps_arrivée_avalanche:.1f} s, t_fin = {temps_fin_avalanche:.1f} s.")
fig.savefig(images_dir / f"puissance-énergie-avalanche{period_return}.png", dpi = 300, bbox_inches = 'tight')

In [ ]:
# calcul des flux selon le modèle AVAC et selon le modèle Wave
avac_frame_temps, wave_frame_temps, flux_volume, flux_volume_eau, volume_lac, volume_lac_eau, masque_lac = \
    calculate_inflow(contour_dict,config_total,verbosity = False)
surface_lac_masque = len(masque_lac[masque_lac==True])*(xmax-xmin)*(ymax-ymin)/(nb_grille-1)**2
print(f"surface du masque autour du lac {format_m(surface_lac_masque)} m².")

In [ ]:
# Comparaison des débits entrant dans le lac (au niveau du contour du lac défini par mask_shp) et du volume
volume_fourni_avalanche   = np.array(list(accumulate(flux_volume, add)))*dt
volume_fourni_avalanche_final = np.trapz(flux_volume,avac_frame_temps)
fig, ((ax1, ax2)) = plt.subplots(1,2)
fig.suptitle(f"Débit entrant et volume d'équivalent en eau fourni par l'avalanche")
fig.set_figheight(4)
fig.set_figwidth(8)
ax1.plot(avac_frame_temps,flux_volume,label = r'$Q$ selon avac')
ax1.plot(wave_frame_temps,flux_volume_eau,label = r'$Q$ selon wave')
ax1.set_ylabel(r"Débit entrant dans le lac [m³/s]")
ax1.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2)

ax2.plot(avac_frame_temps,volume_fourni_avalanche,label = r'$V$ selon avac par intégration')
ax2.plot(avac_frame_temps,volume_lac,label = r'$V$ selon avac')
ax2.plot(wave_frame_temps,volume_lac_eau-volume_0,label = r'$V_{eau,lac}$')
ax2.set_ylabel(r"Volume fourni par l'avalanche [m³]")
ax2.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2)

for ax in (ax1,ax2):
    ax.grid()
    ax.legend(loc='upper right')
    ax.set_xlabel(r"$t$ [s]")
plt.tight_layout()

In [ ]:
plt.close('all')
%matplotlib inline

In [ ]:
# calcul des flux selon le modèle AVAC et selon le modèle Wave
avac_frame_temps, wave_frame_temps, flux_volume, flux_volume_eau, volume_lac, volume_lac_eau, masque_lac = \
    calculate_inflow(contour_dict,config_total,verbosity = False)
surface_lac_masque = len(masque_lac[masque_lac==True])*dx_mnt*dy_mnt
print(f"surface du masque autour du lac {format_m(surface_lac_masque)} m², soit une couverture {surface_lac_masque/surface_0*100:.1f} %")

## Voellmy vs Manning-Strickler

In [ ]:
h = np.array(h_dict['right'])
q = np.array(q_dict['right'])
v = np.zeros(len(h_dict['right']))

np.divide(q,  h, out = v, where = h != 0)
ξ   = avac_config['rheology']['xi']
μ   = avac_config['rheology']['mu']
ρ_a = avac_config['rheology']['rho']
ρ   = parameters['rheology']['rho']
K   = parameters['rheology']['Strickler']
K = 60
d_a = ρ_a/ρ
g   = 9.81
tau_voellmy = μ*ρ_a*g*h+ρ_a*g*v**2/ξ
tau_manning = ρ*g*v**2/(h/d_a)**(1/3)/K**2
plt.plot(tau_voellmy/1e6,tau_manning/1e6)
plt.plot([0,2],[0,2])

# Flux sortant du domaine de calcul

## Flux de masse sortant

In [ ]:
## flux de masse sortant par intégration de la composante normal de hu par la méthode des rectangles ##
# le flux est > 0 si sortant et < 0  si sortant
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(8)
fig.set_figwidth(8)

flux_masse  = {'est'   : np.array([boundary_values('est', frames[i], config)[3].sum()*dy_mnt for i in range(0,NbSim)]),
     'sud'  : -np.array([boundary_values('sud', frames[i], config)[3].sum()*dx_mnt for i in range(0,NbSim)]),
     'nord'  : np.array([boundary_values('nord', frames[i], config)[3].sum()*dx_mnt for i in range(0,NbSim)]),
     'ouest' : -np.array([boundary_values('ouest', frames[i], config)[3].sum()*dy_mnt for i in range(0,NbSim)]),
     }

dict_boundary = {'sud':'(c) sud', 'est':'(a) est', 'nord':'(d) nord', 'ouest':'(b) ouest'}

boundaries = ('est','ouest','sud','nord')
axes_list =  (ax1,ax2,ax3,ax4)
for b, ax in zip(boundaries,axes_list):
    ax.plot(temps_in,flux_masse[b])
    ax.grid()
    ax.text(0.9, 0.9, dict_boundary[b],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
    print(f"* frontière {b} : Qp = {abs(flux_masse[b]).max():.1f} m³/s et V = {flux_masse[b].sum()*dt:.0f} m³.")
for ax in (ax3,ax4):
    ax.set_xlabel(r"$t$ (s)")
for ax in (ax1,ax3):
    ax.set_ylabel(r"$Q$ (m³/s)")
plt.tight_layout()
fig.savefig(images_dir / "flux_masses_vagues.png",**figure_export_params)


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
flux_masse_total = (flux_masse['sud']+flux_masse['nord']+flux_masse['est']+flux_masse['ouest'])
ax.plot(temps_in, flux_masse_total,label=r'$E$')

ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r'Flux total   [m$^3$/s]')
ax.set_xlabel(r'temps $t$ [s]')  
#fig.legend(loc="upper left",ncol=2)

jupyprint(f"Le flux de masse est {format_m(flux_masse_total.sum())}"+" $\\text{m}^3 $.")

## Flux sortant d'énergie cinétique et d'énergie potentielle 

In [ ]:
### Énergie cinétique sortant du domaine

In [ ]:
boundary_values('est', frames[i], config)[2].shape

In [ ]:
# Valeur moyenne du flux d'énergie cinétique sur chaque frontière
# Le flux est < 0 si le flux d'énergie est entrant et > 0 si le flux est sortant
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(8)
fig.set_figwidth(8)

énergie_cinétique_non_masquée = énergie_dic['frames_domaain']['cinétique']

## dictionnaire ##
flux_énergie  = {
    'est'   : np.array([(boundary_values('est', frames[i], config)[2]*énergie_cinétique_non_masquée[i][:,-1]).mean() for i in range(NbSim)]) ,
     'sud'  : -np.array([(boundary_values('sud', frames[i], config)[2]*énergie_cinétique_non_masquée[i][0,:]).mean() for i in range(NbSim)]),
     'nord' : np.array([(boundary_values('nord', frames[i], config)[2]*énergie_cinétique_non_masquée[i][-1,:]).mean() for i in range(NbSim)]),
     'ouest': -np.array([(boundary_values('ouest', frames[i], config)[2]*énergie_cinétique_non_masquée[i][:,0]).mean() for i in range(NbSim)])
     }

dict_boundary = {'sud':'(a) sud', 'est':'(b) est', 'nord':'(c) nord', 'ouest':'(d) ouest'}



boundaries = ('est','ouest','sud','nord')
axes_list =  (ax1,ax2,ax3,ax4)
for b, ax in zip(boundaries,axes_list):
    ax.plot(temps_in,flux_énergie[b]/1e3)
    ax.grid()
    ax.text(0.9, 0.9, dict_boundary[b],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
for ax in (ax3,ax4):
    ax.set_xlabel(r"$t$ (s)")
for ax in (ax1,ax3):
    ax.set_ylabel(r"$\dot E_c$ [kW/m]")
plt.tight_layout()
fig.savefig(images_dir / "flux_énergie_vagues.png",**figure_export_params)

In [ ]:
# Valeur moyenne du flux d'énergie potentielle sur chaque frontière
# Le flux est < 0 si le flux d'énergie est entrant et > 0 si le flux est sortant
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(10)
fig.set_figwidth(10)

## dictionnaire  ##
flux_potentielle  = {
    'est'   : rho*g*np.array([(boundary_values('est', frames[i], config)[3]* \
                                (sol_0[:,-1]+hauteur_non_masquée[i][:,-1]-z_lac)).mean() for i in range(0,NbSim)]) ,
     'sud'  : -rho*g*np.array([(boundary_values('sud', frames[i], config)[3]* \
                                (sol_0[0,:]+hauteur_non_masquée[i][0,:]-z_lac)).mean() for i in range(0,NbSim)]) ,
     'nord' : rho*g*np.array([(boundary_values('nord', frames[i], config)[3]* \
                                (sol_0[-1,:]+hauteur_non_masquée[i][-1,:]-z_lac)).mean() for i in range(0,NbSim)]) ,
     'ouest': -rho*g*np.array([(boundary_values('ouest', frames[i], config)[3]* \
                                (sol_0[:,0]+hauteur_non_masquée[i][:,0]-z_lac)).mean() for i in range(0,NbSim)]) 
     }

dict_boundary = {'sud':'(a) sud', 'est':'(b) est', 'nord':'(c) nord', 'ouest':'(d) ouest'}

boundaries = ('est','ouest','sud','nord')
axes_list =  (ax1,ax2,ax3,ax4)
for b, ax in zip(boundaries,axes_list):
    ax.plot(temps_in,flux_potentielle[b]/1e6)
    ax.grid()
    ax.text(0.9, 0.9, dict_boundary[b],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
for ax in (ax3,ax4):
    ax.set_xlabel(r"$t$ (s)")
for ax in (ax1,ax3):
    ax.set_ylabel(r"$\dot E_p $ [MW/m]")
plt.tight_layout()
fig.savefig(images_dir / "flux_énergie_potentielle_vagues.png",dpi = 300, bbox_inches = 'tight')

In [ ]:
# Valeur totale du flux d'énergie cinétique sur l'ensemble
# Le flux est < 0 si le flux d'énergie est entrant et > 0 si le flux est sortant

# intégration du flux
flux_cinétique_total   = (xmax-xmin)*(flux_énergie['sud'] + flux_énergie['nord']) \
                       + (ymax-ymin)*(flux_énergie['est'] + flux_énergie['ouest'])
flux_potentielle_total = (xmax-xmin)*(flux_potentielle['sud'] + flux_potentielle['nord']) \
                       + (ymax-ymin)*(flux_potentielle['est'] + flux_potentielle['ouest'])

flux_énergie_total     = flux_cinétique_total + flux_potentielle_total

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(temps_in, flux_énergie_total/1e6,label=r'$E$')

ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r"Flux d'énergie   [MW]")
ax.set_xlabel(r'temps $t$ [s]')  
#fig.legend(loc="upper left",ncol=2)
